<a href="https://colab.research.google.com/github/beingAnujChaudhary/Practical-Statistics-for-DS/blob/main/chapter_05_classification/exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

# Mount Google Drive (optional)
drive.mount('/content/drive')

# Clone your GitHub repository
!git clone https://github.com/beingAnujChaudhary/Practical-Statistics-for-DS.git

# Move into repository
%cd /content/Practical-Statistics-for-DS

# Move into Chapter 5 folder
%cd chapter_05_classification

Mounted at /content/drive
Cloning into 'Practical-Statistics-for-DS'...
remote: Enumerating objects: 188, done.
remote: Counting objects: 100% (188/188), done.
remote: Compressing objects: 100% (149/149), done.
Receiving objects: 100% (188/188), 623.93 KiB | 3.06 MiB/s, done.
remote: Total 188 (delta 114), reused 69 (delta 37), pack-reused 0 (from 0)
Resolving deltas: 100% (114/114), done.
/content/Practical-Statistics-for-DS
/content/Practical-Statistics-for-DS/chapter_05_classification


# Chapter 05 Exercises: Classification

> Source: *Practical Statistics for Data Scientists*  
> Chapter 5: Classification

## Goals

In this notebook, I will practise:

- classification basics
- logistic regression
- probabilities
- threshold tuning
- confusion matrix
- accuracy evaluation
- precision, recall, F1-score
- ROC-AUC
- Naive Bayes
- KNN
- class imbalance
- model comparison

---

## Instructions

For each exercise:

1. Predict result first
2. Run the code
3. Interpret findings
4. Explain business meaning

---

```python
# Standard imports for Chapter 5
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve,
    accuracy_score, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score,
    roc_auc_score
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8")
np.random.seed(42)

print("Exercise notebook ready")
```


---

# Exercise 1: Classification vs Regression

## Question

Decide: **classification** or **regression**?

### Scenario 1
Predict: flight delay minutes

### Scenario 2
Predict: delayed or on-time

### Scenario 3
Predict: ticket price

### Scenario 4
Predict: fraud or not fraud

## Write your answers below


---

# Exercise 2: Create Classification Dataset

## Scenario

Predict: flight delay

Features:
- distance
- weather score
- airport congestion

<details>
<summary>Click to view solution</summary>
```python
airline_data = pd.DataFrame({
    "Distance": np.random.normal(1000, 300, 1000),
    "Weather_Score": np.random.normal(5, 2, 1000),
    "Airport_Congestion": np.random.normal(50, 15, 1000)
})

delay_probability = (
    0.002 * airline_data["Distance"] +
    0.08 * airline_data["Weather_Score"] +
    0.01 * airline_data["Airport_Congestion"]
)

airline_data["Delayed"] = (
    np.random.rand(1000) < (delay_probability / delay_probability.max())
).astype(int)

airline_data.head()
```
</details>

## Reflection

1. What are features?
2. What is target variable?
3. Why is this classification?


---

# Exercise 3: Train Logistic Regression Model

<details>
<summary>Click to view solution</summary>
```python
X = airline_data[["Distance", "Weather_Score", "Airport_Congestion"]]
y = airline_data["Delayed"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)
```
</details>

## Reflection

1. Was accuracy high?
2. Is accuracy enough?
3. What problems can accuracy hide?


---

# Exercise 4: Probability Predictions

## Task

Instead of hard predictions, view probabilities.

<details>
<summary>Click to view solution</summary>
```python
probabilities = model.predict_proba(X_test)
probabilities[:5]

probability_df = pd.DataFrame({
    "Actual": y_test.values[:10],
    "Prediction": predictions[:10],
    "Delay Probability": probabilities[:10, 1]
})
probability_df
```
</details>

## Interpretation

Example: `[0.25, 0.75]` means 25% on-time, 75% delayed.

## Reflection

1. Why are probabilities useful?
2. Why not directly predict classes?
3. When does uncertainty matter?



---

# Exercise 5: Threshold Tuning

Default threshold: 0.50. Try: 0.70. How do predictions change?

<details>
<summary>Click to view solution</summary>
```python
custom_predictions = (probabilities[:, 1] > 0.70).astype(int)

custom_accuracy = accuracy_score(y_test, custom_predictions)
print("Custom Accuracy:", custom_accuracy)
```
</details>

## Reflection

1. Did accuracy improve?
2. Did model become stricter?
3. Why might fraud detection use high threshold?


---

# Exercise 6: Confusion Matrix

Analyse model mistakes.

<details>
<summary>Click to view solution</summary>
```python
cm = confusion_matrix(y_test, predictions)
print("Confusion Matrix:\n", cm)

display = ConfusionMatrixDisplay(confusion_matrix=cm)
display.plot()
plt.title("Confusion Matrix")
plt.show()
```
</details>

## Reflection

1. Which mistake happened most?
2. Why are false negatives dangerous?
3. Why are false positives expensive?


---

# Exercise 7: Accuracy Trap

## Scenario

Suppose 99% flights are on-time. Model predicts always on-time.

1. What is accuracy?
2. Is model useful?
3. Why can accuracy mislead?

---

# Exercise 8: Precision

When model predicts positive, how often is it correct? **TP / (TP + FP)**

<details>
<summary>Click to view solution</summary>
```python
precision = precision_score(y_test, predictions)
print("Precision:", precision)
```
</details>

## Reflection

1. Was precision high?
2. Why does precision matter?
3. Which industries care about precision?

### Business Scenario
Fraud detection: false positive means safe customer blocked. Why might precision matter?



---

# Exercise 9: Recall

How many real positives did model detect? **TP / (TP + FN)**

<details>
<summary>Click to view solution</summary>
```python
recall = recall_score(y_test, predictions)
print("Recall:", recall)
```
</details>

## Reflection

1. Was recall high?
2. Why can low recall be dangerous?
3. Which problems require high recall?

### Business Scenario
Medical diagnosis: false negative means missed disease. Why is recall important?


---

# Exercise 10: F1 Score

F1 balances precision and recall. Useful when class imbalance exists.

<details>
<summary>Click to view solution</summary>
```python
f1 = f1_score(y_test, predictions)
print("F1 Score:", f1)
```
</details>

## Reflection

1. Why balance precision and recall?
2. Why can accuracy mislead?
3. When is F1 useful?


---

# Exercise 11: Classification Report

View complete metrics report.

<details>
<summary>Click to view solution</summary>
```python
print(classification_report(y_test, predictions))
```
</details>

## Reflection

1. Which metric looked strongest?
2. Which class struggled?
3. Why should we look beyond accuracy?



---

# Exercise 12: ROC Curve

<details>
<summary>Click to view solution</summary>
```python
probabilities = model.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, probabilities)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr)
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.show()
```
</details>

## Reflection

1. Why is curve above diagonal?
2. What would random guessing look like?
3. Why does ROC matter?


---

# Exercise 13: AUC Score

Range: 0.5 → random, 1.0 → perfect.

<details>
<summary>Click to view solution</summary>
```python
auc = roc_auc_score(y_test, probabilities)
print("AUC Score:", auc)
```
</details>

## Reflection

1. Was AUC strong?
2. Why is AUC useful?
3. Why better than plain accuracy?


---

# Exercise 14: Precision-Recall Curve & Threshold Tuning

<details>
<summary>Click to view solution</summary>
```python
precision, recall, pr_thresholds = precision_recall_curve(y_test, probabilities)

plt.figure(figsize=(6, 4))
plt.plot(recall, precision, color='blue', lw=2, label='PR Curve')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.show()

# Find threshold that maximizes F1
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
best_idx = np.argmax(f1_scores)
best_threshold = pr_thresholds[best_idx]
print(f"Optimal Threshold for F1: {best_threshold:.3f}")

y_pred_opt = (probabilities >= best_threshold).astype(int)
print("\nClassification Report (Optimized Threshold):\n", classification_report(y_test, y_pred_opt))
```
</details>


---

# Exercise 15: Naive Bayes

<details>
<summary>Click to view solution</summary>
```python
naive_bayes_model = GaussianNB()
naive_bayes_model.fit(X_train, y_train)

nb_predictions = naive_bayes_model.predict(X_test)

print("Naive Bayes Accuracy:", accuracy_score(y_test, nb_predictions))
```
</details>

## Reflection

1. How did Naive Bayes perform?
2. Why is it called "naive"?
3. When is Naive Bayes useful?



---

# Exercise 16: K-Nearest Neighbours (KNN)

<details>
<summary>Click to view solution</summary>
```python
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)

knn_predictions = knn_model.predict(X_test)

print("KNN Accuracy:", accuracy_score(y_test, knn_predictions))
```
</details>

## Reflection

1. Did KNN outperform logistic regression?
2. Why might neighbours work?
3. What happens if K becomes too small?


---

# Exercise 17: Class Imbalance

Suppose 95% on-time, 5% delayed. Why is classification difficult?

<details>
<summary>Click to view solution</summary>
```python
imbalanced_target = np.random.choice([0, 1], size=1000, p=[0.95, 0.05])
pd.Series(imbalanced_target).value_counts()

sns.countplot(x=imbalanced_target)
plt.title("Imbalanced Dataset")
plt.show()

# Handling with balanced class weights
lr_balanced = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_balanced.fit(X_train, y_train)

y_pred_bal = lr_balanced.predict(X_test)
print("\nBalanced Model Confusion Matrix:\n", confusion_matrix(y_test, y_pred_bal))
print("\nBalanced Model Classification Report:\n", classification_report(y_test, y_pred_bal))
```
</details>

## Reflection

1. Why can accuracy fail here?
2. Which metrics matter more?
3. Why is fraud detection difficult?



---

# Exercise 18: Model Comparison

Compare Logistic Regression, Naive Bayes, and KNN.

<details>
<summary>Click to view solution</summary>
```python
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Naive Bayes", "KNN"],
    "Accuracy": [
        accuracy_score(y_test, predictions),
        accuracy_score(y_test, nb_predictions),
        accuracy_score(y_test, knn_predictions)
    ]
})
print(comparison)

sns.barplot(data=comparison, x="Model", y="Accuracy")
plt.title("Model Comparison")
plt.show()
```
</details>

## Reflection

1. Which model performed best?
2. Is best accuracy always best model?
3. What trade-offs matter?


---

# Exercise 19: Data Preparation & Logistic Regression with Statsmodels

<details>
<summary>Click to view solution</summary>
```python
# Load data (assumes 'loan_data.csv' is available)
df = pd.read_csv('loan_data.csv')
df['defaulted'] = (df['status'] == 'default').astype(int)
df_clean = df.dropna()

features = ['amount', 'duration', 'interest_rate', 'income']
X = df_clean[features]
y = df_clean['defaulted']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Using statsmodels for interpretable summary
X_train_sm = sm.add_constant(X_train)
model_sm = sm.Logit(y_train, X_train_sm).fit(disp=0)
print(model_sm.summary())
```
</details>


---

# Exercise 20: Statsmodels Predictions & Confusion Matrix

<details>
<summary>Click to view solution</summary>
```python
X_test_sm = sm.add_constant(X_test)
y_prob = model_sm.predict(X_test_sm)
y_pred = (y_prob >= 0.5).astype(int)

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n", classification_report(y_test, y_pred))
```
</details>


---

# Exercise 21: ROC Curve & AUC for Loan Data

<details>
<summary>Click to view solution</summary>
```python
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.show()
```
</details>


---

# Mini Exercise

Suppose model predicts 0.91.

1. What does this mean?
2. Is prediction guaranteed?
3. Why does probability matter?

---

# Interview Questions

1. Difference between classification and regression?
2. What is logistic regression?
3. What is precision?
4. What is recall?
5. Difference between precision and recall?
6. Why can accuracy be misleading?
7. What is class imbalance?
8. What is ROC-AUC?
9. Difference between KNN and logistic regression?
10. Why use probability predictions?

---

# Challenge Problems

## Challenge 1
Build a spam classifier.

## Challenge 2
Experiment with thresholds: 0.30, 0.50, 0.80. Observe precision and recall.

## Challenge 3
Compare KNN vs Naive Bayes vs Logistic Regression.

## Challenge 4
Create an imbalanced dataset. Observe accuracy failure.

## Challenge 5
Try different K values. Observe underfitting vs overfitting.

---

# Progress Checklist

- [ ] Understood classification
- [ ] Used logistic regression
- [ ] Understood probabilities
- [ ] Explored thresholds
- [ ] Used confusion matrix
- [ ] Learned accuracy limitations
- [ ] Learned precision & recall
- [ ] Understood F1-score
- [ ] Used ROC-AUC
- [ ] Used Naive Bayes
- [ ] Used KNN
- [ ] Understood class imbalance
- [ ] Compared models
- [ ] Ready for Chapter 6